# Tiền xử lý & Làm sạch Dữ liệu Google Trends (Clean Trends)

**Mục tiêu của notebook:**
1. Đọc và chuẩn hóa dữ liệu từ khóa Google Trends từ `data-raw/google-trends/trend_topics.parquet`.
2. Phân tích cấu trúc phân cấp phức tạp từ `data-raw/google-trends/trends.json`.
3. Tách và chuẩn hóa thành 6 bảng dữ liệu dạng bảng quan hệ (.parquet) đặt trong `data-cleaned/google-trends/`:
   - `trend_topics.parquet`: Danh mục từ khóa thực thể tuyển sinh đã làm sạch.
   - `trend_queries.parquet`: Danh mục 879 truy vấn đại diện và chỉ số tổng quan.
   - `trend_keywords.parquet`: Bảng ánh xạ quan hệ 1-N giữa từ khóa tuyển sinh và `query_id`.
   - `trend_timeline.parquet`: Chuỗi thời gian điểm quan tâm theo tháng (2004 - 2026).
   - `trend_annual.parquet`: Tổng hợp chỉ số quan tâm theo năm và mùa vụ tuyển sinh (tháng 5-8).
   - `trend_regions.parquet`: Phân bố quan tâm theo 63 tỉnh/thành.
   - `trend_related_queries.parquet`: Truy vấn liên quan phổ biến (Top) và đột biến (Rising).
4. Sử dụng cơ chế `# export` để xuất tự động sang unit script `scripts-processing/unit-scripts/clean_trends.py`.

# 1. Khởi tạo môi trường, Cấu hình đường dẫn & Helper

In [5]:
# export
import json
from pathlib import Path
import sys
import pandas as pd

# Thêm thư mục gốc vào sys.path để nạp helper
PROJECT_ROOT = Path('.').resolve().parents[1] if Path('.').resolve().name == 'notebooks' else Path('.').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Cấu hình UTF-8 an toàn trên Windows
if sys.platform == 'win32':
    for stream in (sys.stdout, sys.stderr):
        if hasattr(stream, 'reconfigure'):
            try:
                stream.reconfigure(encoding='utf-8')
            except Exception:
                pass

from helpers.get_configs import get_config

# Cấu hình đường dẫn dữ liệu thô (Raw Data)
RAW_TOPICS_PATH = get_config('dirs.yaml', ['raw_data', 'trend_topics'], as_path=True)
RAW_TRENDS_PATH = get_config('dirs.yaml', ['raw_data', 'trends'], as_path=True)

# Cấu hình đường dẫn dữ liệu làm sạch (Cleaned Data)
CLEANED_TOPICS_PATH = get_config('dirs.yaml', ['cleaned_data', 'trend_topics'], as_path=True)
CLEANED_QUERIES_PATH = get_config('dirs.yaml', ['cleaned_data', 'trend_queries'], as_path=True)
CLEANED_KEYWORDS_PATH = get_config('dirs.yaml', ['cleaned_data', 'trend_keywords'], as_path=True)
CLEANED_TIMELINE_PATH = get_config('dirs.yaml', ['cleaned_data', 'trend_timeline'], as_path=True)
CLEANED_ANNUAL_PATH = get_config('dirs.yaml', ['cleaned_data', 'trend_annual'], as_path=True)
CLEANED_REGIONS_PATH = get_config('dirs.yaml', ['cleaned_data', 'trend_regions'], as_path=True)
CLEANED_RELATED_PATH = get_config('dirs.yaml', ['cleaned_data', 'trend_related_queries'], as_path=True)

# Cấu hình đường dẫn script đầu ra
SCRIPT_OUTPUT_PATH = get_config('dirs.yaml', ['unit_scripts', 'clean_trends'], as_path=True)


def clean_str(val):
    """Chuẩn hóa chuỗi văn bản: xóa khoảng trắng thừa, xử lý giá trị rỗng."""
    if val is None or pd.isna(val):
        return ""
    return " ".join(str(val).split())

In [6]:
# Khảo sát dữ liệu thô
print("Đường dẫn tệp thô:")
print(f" - RAW_TOPICS_PATH: {RAW_TOPICS_PATH}")
print(f" - RAW_TRENDS_PATH: {RAW_TRENDS_PATH}")

df_raw_topics = pd.read_parquet(RAW_TOPICS_PATH)
print(f"\nTổng số dòng trong trend_topics thô: {len(df_raw_topics):,}")
display(df_raw_topics.head())

Đường dẫn tệp thô:
 - RAW_TOPICS_PATH: D:\UNIDOCS\IS405.R11-Big-Data\data-raw\google-trends\trend_topics.parquet
 - RAW_TRENDS_PATH: D:\UNIDOCS\IS405.R11-Big-Data\data-raw\google-trends\trends.json

Tổng số dòng trong trend_topics thô: 3,579


,keyword,keyword_type,mid,title,type
0,Trường Cao Đẳng Kỹ Thuật Cao Thắng,school,/g/122jr9l6,Trường Cao đẳng Kỹ thuật Cao Thắng,"Trường cao đẳng nghề ở Thành phố Hồ Chí Minh, ..."
1,Trường Cao Đẳng Kỹ Thuật Cao Thắng,school,/g/11c6v9lswf,Thư viện trường Cao đẳng kỹ thuật Cao Thắng,"Thư viện đại học ở Thành phố Hồ Chí Minh, Việt..."
2,Trường Cao Đẳng Kỹ Thuật Cao Thắng,school,/g/11gw35hb5j,Ga Hàm Nghi,Chủ đề
3,Trường Cao Đẳng Công Thương TPHCM,school,/m/0vb4j37,Cao Đẳng Công Thương TP.HCM,"Cao đẳng ở Thành phố Hồ Chí Minh, Việt Nam"
4,Trường Cao Đẳng Công Thương TPHCM,school,/m/0vb3q34,Trường Đại học Công Thương TP. Hồ Chí Minh (HUIT),"Đại học ở Thành phố Hồ Chí Minh, Việt Nam"


# 2. Làm sạch Bảng `trend_topics.parquet`

In [7]:
# export
def clean_trend_topics():
    """Làm sạch, chuẩn hóa ký tự trắng và loại bỏ bản ghi trùng lặp trong trend_topics.parquet."""
    print("--- 1. Làm sạch trend_topics.parquet ---")
    df_raw = pd.read_parquet(RAW_TOPICS_PATH)
    for col in df_raw.columns:
        df_raw[col] = df_raw[col].apply(clean_str)

    df_raw = df_raw[df_raw["keyword"] != ""].copy()
    df_clean = df_raw.drop_duplicates(subset=["keyword", "keyword_type", "mid", "title"]).reset_index(drop=True)

    CLEANED_TOPICS_PATH.parent.mkdir(parents=True, exist_ok=True)
    df_clean.to_parquet(CLEANED_TOPICS_PATH, index=False)
    print(f"Đã lưu cleaned trend_topics.parquet: {len(df_clean):,} dòng (từ {len(df_raw):,} dòng thô).")
    return df_clean

In [8]:
# Thực thi làm sạch trend_topics.parquet
df_cleaned_topics = clean_trend_topics()
df_cleaned_topics.info()
df_cleaned_topics.head()

--- 1. Làm sạch trend_topics.parquet ---
Đã lưu cleaned trend_topics.parquet: 3,579 dòng (từ 3,579 dòng thô).
<class 'pandas.DataFrame'>
RangeIndex: 3579 entries, 0 to 3578
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   keyword       3579 non-null   str  
 1   keyword_type  3579 non-null   str  
 2   mid           3579 non-null   str  
 3   title         3579 non-null   str  
 4   type          3579 non-null   str  
dtypes: str(5)
memory usage: 529.1 KB


,keyword,keyword_type,mid,title,type
0,Trường Cao Đẳng Kỹ Thuật Cao Thắng,school,/g/122jr9l6,Trường Cao đẳng Kỹ thuật Cao Thắng,"Trường cao đẳng nghề ở Thành phố Hồ Chí Minh, ..."
1,Trường Cao Đẳng Kỹ Thuật Cao Thắng,school,/g/11c6v9lswf,Thư viện trường Cao đẳng kỹ thuật Cao Thắng,"Thư viện đại học ở Thành phố Hồ Chí Minh, Việt..."
2,Trường Cao Đẳng Kỹ Thuật Cao Thắng,school,/g/11gw35hb5j,Ga Hàm Nghi,Chủ đề
3,Trường Cao Đẳng Công Thương TPHCM,school,/m/0vb4j37,Cao Đẳng Công Thương TP.HCM,"Cao đẳng ở Thành phố Hồ Chí Minh, Việt Nam"
4,Trường Cao Đẳng Công Thương TPHCM,school,/m/0vb3q34,Trường Đại học Công Thương TP. Hồ Chí Minh (HUIT),"Đại học ở Thành phố Hồ Chí Minh, Việt Nam"


# 3. Phân tách và Chuẩn hóa `trends.json` thành 6 Bảng Quan hệ Parquet

Tệp thô `data-raw/google-trends/trends.json` là một danh mục các JSON Object có cấu trúc phân cấp, đa tầng và lồng ghép sâu (nested objects/arrays), phản ánh 3 widget dữ liệu từ Google Trends (`interest_over_time`, `interest_by_region`, `commonly_searched_queries`).

Để phục vụ phân tích dữ liệu dạng bảng (Tabular Analysis), xây dựng đặc trưng chuỗi thời gian và huấn luyện mô hình Machine Learning, chúng ta **chuẩn hóa quan hệ (Relational Normalization)** tệp JSON này thành **6 bảng Parquet quan hệ độc lập** đặt trong `data-cleaned/google-trends/`.

---

### 3.1 Chi tiết 6 Bảng Quan hệ, Schema và Phương pháp Trích xuất

#### 1. Bảng `trend_queries.parquet` (879 dòng) - Thực thể Truy vấn Đại diện
- **Bản chất nghiệp vụ:** Bảng thực thể chính (Primary Entity Table), quản lý 879 truy vấn đại diện đã được gửi đến Google Trends (gồm 559 Knowledge Graph MID và 320 từ khóa tìm kiếm fallback). Mỗi bản ghi có một khóa chính duy nhất `query_id`.
- **Phương pháp trích xuất:**
  - Trích xuất các thuộc tính cấp cao nhất (root-level properties) của từng object trong `trends.json`: `query_id`, `query`, `query_kind`, `mid`, `title`, `type`, `selection_score`, `status`, `geo`, `date_range`.
  - Bóc tách trạng thái từng giai đoạn thu thập từ dictionary `stage_status` (`stage_interest`, `stage_enrichment`, `enrichment_skip_reason`).
  - Lấy các chỉ số thống kê tổng hợp toàn chuỗi thời gian từ object `interest_over_time`: `nonzero_points`, `average_interest`, `max_interest`.
- **Schema tóm tắt:**
  - `query_id` (`str`): Khóa chính định danh duy nhất (SHA-256 hash 20 ký tự).
  - `query` (`str`): Chuỗi truy vấn gửi lên Google Trends (MID hoặc keyword).
  - `query_kind` (`str`): Phân loại truy vấn (`topic_mid` hoặc `search_term`).
  - `mid`, `title`, `type` (`str`): Định danh thực thể do Google Knowledge Graph phân loại.
  - `selection_score` (`float`): Điểm độ tương đồng ngữ nghĩa khi lựa chọn topic (0.0 đến 1.0).
  - `status`, `stage_interest`, `stage_enrichment` (`str`): Trạng thái cào dữ liệu (`success`, `interest_only`, `skipped`).
  - `nonzero_points` (`int`): Số tháng có lượt tìm kiếm $> 0$.
  - `average_interest` (`float`), `max_interest` (`int`): Điểm quan tâm trung bình và đỉnh điểm toàn thời gian (0 - 100).
  - `geo` (`str`), `date_range` (`str`): Vùng địa lý (*VN*) và dải thời gian (*all* = 2004 - 2026).

---

#### 2. Bảng `trend_keywords.parquet` (927 dòng) - Ánh xạ Thực thể Tuyển sinh (1-N Mapping Table)
- **Bản chất nghiệp vụ:** Bảng cầu nối quan hệ (Junction / Entity Mapping Table) liên kết các từ khóa thực thể trong tuyển sinh sang truy vấn Google Trends. Một `query_id` có thể đại diện cho nhiều từ khóa tuyển sinh đồng nghĩa hoặc gộp chung.
- **Phương pháp trích xuất:**
  - Duyệt qua mảng đối tượng `keyword_metadata` trong từng record của `trends.json`.
  - Làm phẳng (Unnest) từng phần tử metadata chứa từ khóa ban đầu và thông tin tuyển chọn topic.
  - Trường hợp không có `keyword_metadata`, giải nén cặp danh sách song song `keywords` và `keyword_types`.
  - Loại bỏ các dòng trùng lặp bằng `.drop_duplicates()`.
- **Schema tóm tắt:**
  - `query_id` (`str`): Khóa ngoại liên kết tới `trend_queries.parquet`.
  - `keyword` (`str`): Tên thực thể gốc trong dữ liệu tuyển sinh (VD: *Đại Học Kinh Tế Quốc Dân, Công nghệ thông tin*).
  - `keyword_type` (`str`): Phân loại thực thể (`school`, `major`, `major_group`, `subject_group`, `subject`, `admission_method`).
  - `selected_mid`, `selected_title` (`str`): Topic Google Trends được chỉ định đại diện cho từ khóa.
  - `relevance_score` (`float`): Điểm tương đồng ngữ nghĩa giữa từ khóa và topic được chọn.
  - `selection_reason` (`str`): Lý do lựa chọn (`best_relevant_topic` hoặc `keyword_fallback`).
  - `candidate_count` (`int`): Số lượng ứng viên topic mà Google gợi ý.

---

#### 3. Bảng `trend_timeline.parquet` (240,846 dòng) - Chuỗi Thời gian Chi tiết theo Tháng
- **Bản chất nghiệp vụ:** Bảng chuỗi thời gian chi tiết (Fine-grained Time Series Table) lưu trữ điểm số quan tâm theo từng tháng từ tháng 01/2004 đến tháng 10/2026 (274 mốc tháng cho mỗi query).
- **Phương pháp trích xuất:**
  - Truy xuất vào dictionary `interest_over_time`, lấy mảng đối tượng `timeline`.
  - Lặp qua từng mốc thời gian `{'date': 'YYYY-MM-DDTHH:MM:SS', 'value': int}`.
  - Chuẩn hóa chuỗi ngày giờ thành định dạng ngày `YYYY-MM-DD`.
  - Tách và chuyển đổi `year` và `month` thành số nguyên (`int`) nhằm tăng tốc độ lọc và join theo năm.
- **Schema tóm tắt:**
  - `query_id` (`str`): Khóa ngoại liên kết tới `trend_queries.parquet`.
  - `date` (`str`): Ngày bắt đầu của tháng quan sát (`YYYY-MM-DD`).
  - `year` (`int`): Năm quan sát (2004 đến 2026).
  - `month` (`int`): Tháng quan sát (1 đến 12).
  - `interest` (`int`): Điểm mức độ quan tâm chuẩn hóa của Google Trends (0 đến 100).

---

#### 4. Bảng `trend_annual.parquet` (20,217 dòng) - Tổng hợp Xu hướng theo Năm & Mùa Tuyển sinh
- **Bản chất nghiệp vụ:** Bảng tổng hợp trước (Pre-aggregated Summary Table) được thiết kế đặc thù để nối trực tiếp với năm tuyển sinh `score_year` trong bảng điểm chuẩn, giúp tăng tốc độ truy vấn gấp hàng trăm lần so với việc tính toán on-the-fly từ chuỗi thời gian 240,000 dòng.
- **Phương pháp trích xuất:**
  - Gom nhóm các điểm dữ liệu trong `timeline` vào các thùng chứa theo từng năm (Yearly Buckets).
  - Tính toán mức quan tâm trung bình năm `interest_mean = sum(values) / len(values)`.
  - Xác định đỉnh tìm kiếm trong năm `interest_max = max(values)` và tổng tích lũy `interest_sum`.
  - **Trích xuất đặc trưng Mùa cao điểm Tuyển sinh:** Lọc riêng các điểm dữ liệu thuộc các tháng 5, 6, 7, 8 (thời điểm thi tốt nghiệp THPT và nộp hồ sơ nguyện vọng tại Việt Nam) để tính chỉ số `interest_season_peak`.
  - Đếm số tháng có phát sinh tìm kiếm khác 0 (`nonzero_months`).
- **Schema tóm tắt:**
  - `query_id` (`str`): Khóa ngoại liên kết tới `trend_queries.parquet`.
  - `year` (`int`): Năm tuyển sinh tương ứng.
  - `interest_mean` (`float`): Điểm quan tâm tìm kiếm trung bình của 12 tháng trong năm.
  - `interest_max` (`int`): Điểm tìm kiếm cực đại trong năm (0 - 100).
  - `interest_sum` (`int`): Tổng điểm quan tâm tích lũy trong năm.
  - `interest_season_peak` (`float`): Điểm quan tâm trung bình trong mùa tuyển sinh (Tháng 5-8).
  - `nonzero_months` (`int`): Số tháng trong năm có lượt tìm kiếm $> 0$.

---

#### 5. Bảng `trend_regions.parquet` (10,195 dòng) - Phân bố Mức độ Quan tâm theo 63 Tỉnh/Thành
- **Bản chất nghiệp vụ:** Bảng phân bố địa lý (Geographical Distribution Table) cho 200 query có lượng tìm kiếm lớn nhất được làm giàu thông tin (Enriched).
- **Phương pháp trích xuất:**
  - Kiểm tra sự tồn tại của object `interest_by_region` trong từng query.
  - Duyệt qua mảng `regions` chứa danh sách 63 tỉnh/thành phố tại Việt Nam.
  - Chuẩn hóa tên tỉnh thành và trích xuất thứ hạng `rank` cùng điểm số quan tâm phân bổ `value`.
- **Schema tóm tắt:**
  - `query_id` (`str`): Khóa ngoại liên kết tới `trend_queries.parquet`.
  - `region` (`str`): Tên tỉnh/thành phố (VD: *Hà Nội, Bến Tre, TP. Hồ Chí Minh*).
  - `rank` (`int`): Thứ hạng quan tâm của tỉnh/thành đối với chủ đề này.
  - `value` (`int`): Tỷ lệ / Điểm số mức độ quan tâm phân bổ tại địa phương (0 - 100).

---

#### 6. Bảng `trend_related_queries.parquet` (1,182 dòng) - Truy vấn Tìm kiếm Liên quan (Top & Rising)
- **Bản chất nghiệp vụ:** Bảng ngữ nghĩa truy vấn liên quan (Semantic Search Queries Table) lưu các từ khóa người dùng tìm kiếm đồng thời khi tìm về trường hoặc ngành học.
- **Phương pháp trích xuất:**
  - Truy xuất vào dictionary `commonly_searched_queries`.
  - Hỗ trợ linh hoạt cả hai định dạng dữ liệu từ scraper (dạng `list` trực tiếp hoặc dictionary nhóm theo query term).
  - Trích xuất danh mục `top_queries` (các từ khóa tìm kiếm nhiều nhất) và gán `query_type = 'top'`, `metric_type = 'score'`.
  - Trích xuất danh mục `rising_queries` (các từ khóa tăng trưởng đột biến) và gán `query_type = 'rising'`, `metric_type = 'change'`.
  - Bóc tách cờ `is_breakout` (`True` nếu mức tăng trưởng $\ge 5000\%$).
- **Schema tóm tắt:**
  - `query_id` (`str`): Khóa ngoại liên kết tới `trend_queries.parquet`.
  - `query_type` (`str`): Phân loại nhóm truy vấn: `top` (phổ biến) hoặc `rising` (đột biến).
  - `related_query` (`str`): Cụm từ người dùng tìm kiếm (VD: *điểm chuẩn bách khoa 2024, học phí bưu chính viễn thông*).
  - `rank` (`int`): Thứ hạng trong danh mục gợi ý của Google.
  - `value` (`str`): Điểm số (nếu là top, 0-100) hoặc tỷ lệ tăng trưởng phần trăm / Breakout (nếu là rising).
  - `is_breakout` (`bool`): Cờ đánh dấu lượt tìm kiếm tăng trưởng phi mã.
  - `metric_type` (`str`): Đơn vị đo lường (`score` hoặc `change`).

In [9]:
# export
def clean_trends_json():
    """Tách và chuẩn hóa trends.json thành các bảng quan hệ Parquet chuyên biệt."""
    print("--- 2. Tách và chuẩn hóa trends.json ---")
    with open(RAW_TRENDS_PATH, "r", encoding="utf-8") as f:
        records = json.load(f)

    queries_list = []
    keywords_list = []
    timeline_list = []
    annual_list = []
    regions_list = []
    related_list = []

    for r in records:
        qid = r.get("query_id")
        q = r.get("query", "")
        q_kind = r.get("query_kind", "")
        mid = r.get("mid", "") or ""
        title = r.get("title", "") or ""
        topic_type = r.get("type", "") or ""
        score = r.get("selection_score", 0.0)
        status = r.get("status", "")
        geo = r.get("geo", "")
        date_range = r.get("date_range", "")

        stage_status = r.get("stage_status", {})
        st_interest = stage_status.get("interest", "")
        st_enrichment = stage_status.get("enrichment", "")
        enrichment_skip_reason = stage_status.get("enrichment_skip_reason", "") or ""

        iot = r.get("interest_over_time") or {}
        nonzero_points = iot.get("nonzero_points", 0)
        avg_interest = iot.get("average_interest", 0.0)
        max_interest = iot.get("max_interest", 0)

        # 1. Bảng trend_queries
        queries_list.append({
            "query_id": qid,
            "query": q,
            "query_kind": q_kind,
            "mid": mid,
            "title": title,
            "type": topic_type,
            "selection_score": score,
            "status": status,
            "stage_interest": st_interest,
            "stage_enrichment": st_enrichment,
            "enrichment_skip_reason": enrichment_skip_reason,
            "nonzero_points": nonzero_points,
            "average_interest": avg_interest,
            "max_interest": max_interest,
            "geo": geo,
            "date_range": date_range,
        })

        # 2. Bảng trend_keywords (Ánh xạ 1-N thực thể tuyển sinh sang query_id)
        kw_meta = r.get("keyword_metadata", [])
        if kw_meta:
            for km in kw_meta:
                keywords_list.append({
                    "query_id": qid,
                    "keyword": clean_str(km.get("keyword")),
                    "keyword_type": clean_str(km.get("keyword_type")),
                    "selected_mid": clean_str(km.get("selected_mid")),
                    "selected_title": clean_str(km.get("selected_title")),
                    "relevance_score": km.get("relevance_score", 0.0),
                    "selection_reason": km.get("selection_reason", ""),
                    "candidate_count": km.get("candidate_count", 1),
                })
        else:
            kws = r.get("keywords", [])
            types = r.get("keyword_types", [])
            for kw, kt in zip(kws, types):
                keywords_list.append({
                    "query_id": qid,
                    "keyword": clean_str(kw),
                    "keyword_type": clean_str(kt),
                    "selected_mid": mid,
                    "selected_title": title,
                    "relevance_score": score,
                    "selection_reason": "default",
                    "candidate_count": 1,
                })

        # 3. Bảng trend_timeline & 4. trend_annual
        timeline = iot.get("timeline", [])
        yearly_buckets = {}
        for pt in timeline:
            raw_date = pt.get("date", "")
            val = pt.get("value", 0)
            date_clean = raw_date[:10]
            year = int(date_clean[:4])
            month = int(date_clean[5:7])

            timeline_list.append({
                "query_id": qid,
                "date": date_clean,
                "year": year,
                "month": month,
                "interest": val,
            })

            if year not in yearly_buckets:
                yearly_buckets[year] = {"values": [], "season_values": [], "nonzero": 0}
            yearly_buckets[year]["values"].append(val)
            if val > 0:
                yearly_buckets[year]["nonzero"] += 1
            if month in (5, 6, 7, 8):  # Mùa cao điểm tuyển sinh
                yearly_buckets[year]["season_values"].append(val)

        for year, bucket in yearly_buckets.items():
            vals = bucket["values"]
            s_vals = bucket["season_values"]
            annual_list.append({
                "query_id": qid,
                "year": year,
                "interest_mean": round(sum(vals) / len(vals), 4) if vals else 0.0,
                "interest_max": max(vals) if vals else 0,
                "interest_sum": sum(vals),
                "interest_season_peak": round(sum(s_vals) / len(s_vals), 4) if s_vals else 0.0,
                "nonzero_months": bucket["nonzero"],
            })

        # 5. Bảng trend_regions (Quan tâm theo tỉnh thành)
        ibr = r.get("interest_by_region")
        if ibr and "regions" in ibr and isinstance(ibr["regions"], list):
            for item in ibr["regions"]:
                regions_list.append({
                    "query_id": qid,
                    "region": clean_str(item.get("region")),
                    "rank": item.get("rank"),
                    "value": item.get("value", 0),
                })

        # 6. Bảng trend_related_queries (Truy vấn liên quan Top & Rising)
        csq = r.get("commonly_searched_queries")
        if csq and isinstance(csq, dict):
            def parse_items(container, q_type):
                raw_items = []
                if isinstance(container, list):
                    raw_items = container
                elif isinstance(container, dict):
                    for sub in container.values():
                        if isinstance(sub, list):
                            raw_items.extend(sub)
                for it in raw_items:
                    if isinstance(it, dict):
                        related_list.append({
                            "query_id": qid,
                            "query_type": q_type,
                            "related_query": clean_str(it.get("query")),
                            "rank": it.get("rank"),
                            "value": str(it.get("value", "")),
                            "is_breakout": bool(it.get("is_breakout", False)),
                            "metric_type": it.get("metric_type", "score" if q_type == "top" else "change"),
                        })

            parse_items(csq.get("top_queries"), "top")
            parse_items(csq.get("rising_queries"), "rising")

    # Chuyển đổi thành DataFrames và lưu vào các file Parquet
    df_queries = pd.DataFrame(queries_list)
    df_keywords = pd.DataFrame(keywords_list).drop_duplicates().reset_index(drop=True)
    df_timeline = pd.DataFrame(timeline_list)
    df_annual = pd.DataFrame(annual_list)
    df_regions = pd.DataFrame(regions_list)
    df_related = pd.DataFrame(related_list)

    CLEANED_QUERIES_PATH.parent.mkdir(parents=True, exist_ok=True)
    df_queries.to_parquet(CLEANED_QUERIES_PATH, index=False)
    df_keywords.to_parquet(CLEANED_KEYWORDS_PATH, index=False)
    df_timeline.to_parquet(CLEANED_TIMELINE_PATH, index=False)
    df_annual.to_parquet(CLEANED_ANNUAL_PATH, index=False)
    df_regions.to_parquet(CLEANED_REGIONS_PATH, index=False)
    df_related.to_parquet(CLEANED_RELATED_PATH, index=False)

    print(f"Đã lưu trend_queries.parquet: {len(df_queries):,} dòng")
    print(f"Đã lưu trend_keywords.parquet: {len(df_keywords):,} dòng")
    print(f"Đã lưu trend_timeline.parquet: {len(df_timeline):,} dòng")
    print(f"Đã lưu trend_annual.parquet: {len(df_annual):,} dòng")
    print(f"Đã lưu trend_regions.parquet: {len(df_regions):,} dòng")
    print(f"Đã lưu trend_related_queries.parquet: {len(df_related):,} dòng")
    return df_queries, df_keywords, df_timeline, df_annual, df_regions, df_related

In [10]:
# Thực thi làm sạch và tách trends.json
dfs = clean_trends_json()
df_queries, df_keywords, df_timeline, df_annual, df_regions, df_related = dfs

--- 2. Tách và chuẩn hóa trends.json ---
Đã lưu trend_queries.parquet: 879 dòng
Đã lưu trend_keywords.parquet: 927 dòng
Đã lưu trend_timeline.parquet: 240,846 dòng
Đã lưu trend_annual.parquet: 20,217 dòng
Đã lưu trend_regions.parquet: 10,195 dòng
Đã lưu trend_related_queries.parquet: 1,182 dòng


# 4. Hàm Khởi chạy cho Unit Script

In [11]:
# export
if __name__ == "__main__":
    clean_trend_topics()
    clean_trends_json()
    print("Hoàn tất làm sạch Google Trends!")

--- 1. Làm sạch trend_topics.parquet ---
Đã lưu cleaned trend_topics.parquet: 3,579 dòng (từ 3,579 dòng thô).
--- 2. Tách và chuẩn hóa trends.json ---
Đã lưu trend_queries.parquet: 879 dòng
Đã lưu trend_keywords.parquet: 927 dòng
Đã lưu trend_timeline.parquet: 240,846 dòng
Đã lưu trend_annual.parquet: 20,217 dòng
Đã lưu trend_regions.parquet: 10,195 dòng
Đã lưu trend_related_queries.parquet: 1,182 dòng
Hoàn tất làm sạch Google Trends!


# 5. Xuất Notebook sang Unit Script (`clean_trends.py`)

In [12]:
from helpers.export_to_script import export_to_script

NOTEBOOK_INPUT_PATH = Path("scripts-processing/notebooks/clean-trends.ipynb")

# Ghi đè unit script scripts-processing/unit-scripts/clean_trends.py
exported_file = export_to_script(
    input_path=NOTEBOOK_INPUT_PATH,
    output_path=SCRIPT_OUTPUT_PATH
)
print(f"Đã xuất thành công sang unit script: {exported_file}")

Đã xuất 4 cell từ 'clean-trends.ipynb' sang 'D:\UNIDOCS\IS405.R11-Big-Data\scripts-processing\unit-scripts\clean_trends.py'
Đã xuất thành công sang unit script: D:\UNIDOCS\IS405.R11-Big-Data\scripts-processing\unit-scripts\clean_trends.py
